## cloning the repo

In [1]:
!git clone https://gitlab.inria.fr/fmajorcz/a_new_hope_for_darpa_optc.git

Cloning into 'a_new_hope_for_darpa_optc'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 394 (delta 7), reused 3 (delta 0), pack-reused 364 (from 1)
Receiving objects: 100% (394/394), 137.09 MiB | 33.97 MiB/s, done.
Resolving deltas: 100% (97/97), done.
Updating files: 100% (371/371), done.


In [2]:
!mv /kaggle/working/a_new_hope_for_darpa_optc/labelling/host/ground_truths/ground_truth_corrected_all_evts.json.gz /kaggle/working/
!rm -rf /kaggle/working/a_new_hope_for_darpa_optc/

In [4]:
!gunzip /kaggle/working/ground_truth_corrected_all_evts.json.gz

In [6]:
import duckdb
import os

json_path = "/kaggle/working/ground_truth_corrected_all_evts.json"
parquet_path = "/kaggle/working/ground_truth_corrected_all_evts.parquet"

con = duckdb.connect()

con.sql(f"""
    COPY (
        SELECT *
        FROM read_json_auto('{json_path}',
            format = 'auto',
            maximum_object_size = 500000000,
            sample_size = 200000,
            ignore_errors = true
        )
    ) TO '{parquet_path}' (
        FORMAT PARQUET,
        COMPRESSION 'ZSTD',
        ROW_GROUP_SIZE 100000
    )
""")

print(f"Conversion done → {parquet_path}")
print(f"Size: {os.path.getsize(parquet_path)/(1024**3):.2f} GB")

# Quick schema peek
con.sql(f"DESCRIBE '{parquet_path}'").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Conversion done → /kaggle/working/ground_truth_corrected_all_evts.parquet
Size: 0.01 GB


,column_name,column_type,null,key,default,extra
0,action,VARCHAR,YES,None,None,None
1,actorID,UUID,YES,None,None,None
2,hostname,VARCHAR,YES,None,None,None
3,id,UUID,YES,None,None,None
4,object,VARCHAR,YES,None,None,None
5,objectID,UUID,YES,None,None,None
6,pid,BIGINT,YES,None,None,None
7,ppid,BIGINT,YES,None,None,None
8,principal,VARCHAR,YES,None,None,None
9,properties,"STRUCT(acuity_level VARCHAR, command_line VARC...",YES,None,None,None


## No of malicious events

In [10]:
con.sql(f"""
    SELECT COUNT(*)
    FROM read_json_auto('{json_path}')
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       360567 │
└──────────────┘

## 360k events
So this file has all the malicious events as mentioned in the [paper](https://inria.hal.science/hal-05474126v1/).

## Malicious event id's

In [14]:
con.sql(f"""
    SELECT id
    FROM read_json_auto('{json_path}')
""")

┌──────────────────────────────────────┐
│                  id                  │
│                 uuid                 │
├──────────────────────────────────────┤
│ f3815ab9-206c-4072-8a50-46a1d62e10f6 │
│ 373f8e64-f391-409c-af2c-e8cf618f49d4 │
│ 27f829fe-0a56-451a-becf-4ddb4ec18021 │
│ 6e1c361d-b3ef-4c6b-a43d-cb27fa55b3f6 │
│ 89f200d1-0095-449f-bc49-8d04cfb5fa3d │
│ e092826f-c336-44d3-bbc6-fedb13c3038f │
│ 2680adab-4106-40d9-ab20-546d53cfeb7d │
│ 1325131a-50ae-4e7e-9ccd-5873239b001c │
│ cff80b9a-6629-433d-9bcd-d6638f53a21c │
│ 8adb488b-b29d-4363-9aae-96e394cec4a8 │
│                  ·                   │
│                  ·                   │
│                  ·                   │
│ a568b899-57b6-43db-a4a8-2d60446bd32b │
│ 82ae38c9-bef8-4d32-a912-f97dcea0dc4e │
│ d4280565-e778-48c9-8af2-8e467265dc20 │
│ c443a43b-8d81-4ddd-9aa6-df72f93681cd │
│ 1767da84-5029-4556-afa1-11bd48ceb57c │
│ f3240ea0-2b8e-4678-af3a-9402b2a83189 │
│ 69fb9688-ccb3-47b9-82b8-c9872f38e245 │
│ e8490d85-0675-